# Analysis Notebook for ACL Paper

This notebook computes statistics and generates tables/figures for the paper.

**Data sources:**
- `Data/narratives/*.csv` - Generated NLEs (9 strategies) - for **efficiency** analysis
- `Data/forsetzung_results/*/geval_*.csv` - G-Eval scores - for **quality** analysis

In [163]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Paths
NARRATIVES_DIR = Path("../Data/narratives")
EVAL_DIR = Path("../Data/forsetzung_results/20250813_135743")

print(f"Narratives dir exists: {NARRATIVES_DIR.exists()}")
print(f"Evaluation dir exists: {EVAL_DIR.exists()}")

Narratives dir exists: True
Evaluation dir exists: True


In [164]:
# Load all narrative files (GENERATION data, not evaluation)
NARRATIVES_DIR = Path("../Data/narratives")

dfs = []
for csv_file in sorted(NARRATIVES_DIR.glob("*.csv")):
    strategy = csv_file.stem  # filename = strategy name
    temp_df = pd.read_csv(csv_file)
    temp_df['Strategy'] = strategy
    dfs.append(temp_df)
    print(f"Loaded {strategy}: {len(temp_df)} rows")

df_all = pd.concat(dfs, ignore_index=True)

# Separate main analysis from temperature sweep
df = df_all[df_all['Strategy'] != 'zero_shot_temp'].copy()
df_temp = df_all[df_all['Strategy'] == 'zero_shot_temp'].copy()

print(f"\n{'='*50}")
print(f"Total rows: {len(df_all):,}")
print(f"Main analysis (excl. temp sweep): {len(df):,}")
print(f"Temperature sweep: {len(df_temp):,}")

print(f"\nFactors in main analysis:")
for col in ['LLM', 'Model', 'XAI', 'Strategy']:
    print(f"  {col}: {df[col].nunique()} levels - {sorted(df[col].unique())}")

Loaded cot_few_shot: 60 rows
Loaded cot_zero_shot: 60 rows
Loaded few_shot: 90 rows
Loaded meta_prompting: 90 rows
Loaded reflexion: 90 rows
Loaded role_based: 90 rows
Loaded self_consistency: 90 rows
Loaded zero_shot: 90 rows
Loaded zero_shot_temp: 1890 rows

Total rows: 2,550
Main analysis (excl. temp sweep): 660
Temperature sweep: 1,890

Factors in main analysis:
  LLM: 3 levels - ['DEEPSEEK', 'GPT', 'L3_LOCAL']
  Model: 4 levels - ['MLP', 'RandomForest', 'SARIMAX', 'XGB']
  XAI: 3 levels - ['lime', 'none', 'shap']
  Strategy: 8 levels - ['cot_few_shot', 'cot_zero_shot', 'few_shot', 'meta_prompting', 'reflexion', 'role_based', 'self_consistency', 'zero_shot']


## 2. Efficiency Table (Table 1 in paper)

Compute Tokens and Time by each experimental factor:
- Strategy (8 prompting strategies)
- XAI (SHAP, LIME, None)
- LLM (GPT-4o, Llama-3, DeepSeek-R1)
- Model (XGBoost, RF, MLP, SARIMAX)

In [165]:
def compute_efficiency_by_factor(data, factor_col):
    """Compute mean ± SD for Tokens and Time by factor level."""
    grouped = data.groupby(factor_col).agg({
        'TokensTotal': ['count', 'mean', 'std'],
        'Duration_s': ['mean', 'std']
    }).round(1)
    
    grouped.columns = ['N', 'Tokens_mean', 'Tokens_std', 'Time_mean', 'Time_std']
    grouped = grouped.reset_index()
    grouped = grouped.rename(columns={factor_col: 'Level'})
    grouped['Factor'] = factor_col
    grouped = grouped[['Factor', 'Level', 'N', 'Tokens_mean', 'Tokens_std', 'Time_mean', 'Time_std']]
    grouped = grouped.sort_values('Tokens_mean')  # Order by tokens ascending
    
    return grouped

# Compute efficiency for each factor (main study)
efficiency_strategy = compute_efficiency_by_factor(df, 'Strategy')
efficiency_xai = compute_efficiency_by_factor(df, 'XAI')
efficiency_llm = compute_efficiency_by_factor(df, 'LLM')
efficiency_model = compute_efficiency_by_factor(df, 'Model')

# Compute efficiency for temperature sweep
efficiency_temp = compute_efficiency_by_factor(df_temp, 'Temperature')
efficiency_temp['Level'] = efficiency_temp['Level'].apply(lambda x: f"{x:.1f}")

# Combine all factors
efficiency_table = pd.concat([
    efficiency_strategy,
    efficiency_xai,
    efficiency_llm,
    efficiency_model,
    efficiency_temp
], ignore_index=True)

# Compute grand mean across ALL data
grand_n = len(df_all)
grand_tokens_mean = df_all['TokensTotal'].mean()
grand_tokens_std = df_all['TokensTotal'].std()
grand_time_mean = df_all['Duration_s'].mean()
grand_time_std = df_all['Duration_s'].std()

print("=" * 80)
print(f"GRAND MEAN: N={grand_n}, Tokens={grand_tokens_mean:.0f}±{grand_tokens_std:.0f}, Time={grand_time_mean:.1f}±{grand_time_std:.1f}")
print("=" * 80)
print("\nEfficiency Table (ordered by Tokens within each factor):")
display(efficiency_table)

GRAND MEAN: N=2550, Tokens=653±1188, Time=35.2±72.0

Efficiency Table (ordered by Tokens within each factor):


,Factor,Level,N,Tokens_mean,Tokens_std,Time_mean,Time_std
0,Strategy,cot_few_shot,60,215.4,39.8,35.2,34.0
1,Strategy,cot_zero_shot,60,279.5,45.7,29.1,27.6
2,Strategy,zero_shot,90,498.6,618.0,22.9,23.7
3,Strategy,few_shot,90,520.4,743.2,29.5,31.8
4,Strategy,meta_prompting,90,525.2,435.0,29.4,32.3
5,Strategy,role_based,90,538.6,605.9,25.1,26.3
6,Strategy,reflexion,90,2184.2,2869.1,132.9,190.2
7,Strategy,self_consistency,90,3717.8,3445.9,219.2,216.8
8,XAI,none,264,919.9,1226.5,62.2,115.5
9,XAI,shap,198,951.8,1181.7,65.8,130.4


In [166]:
def generate_full_latex_efficiency_table(efficiency_df, grand_stats):
    """Generate complete LaTeX efficiency table with grand mean in header."""
    
    # Display name mappings
    level_names = {
        # Strategy
        'zero_shot': 'Zero-shot', 'few_shot': 'Few-shot', 'cot_zero_shot': 'CoT Zero',
        'cot_few_shot': 'CoT Few', 'role_based': 'Role-based', 'meta_prompting': 'Meta-prompt',
        'reflexion': 'Reflexion', 'self_consistency': 'Self-consistency',
        # XAI
        'none': 'None', 'shap': 'SHAP', 'lime': 'LIME',
        # LLM
        'GPT': 'GPT-4o', 'L3_LOCAL': 'Llama-3', 'DEEPSEEK': 'DeepSeek-R1',
        # Model - keep as is
    }
    
    factor_labels = {
        'Strategy': 'Strategy', 'XAI': 'XAI', 'LLM': 'LLM', 
        'Model': 'Model', 'Temperature': 'Temp.'
    }
    
    # Start building LaTeX
    lines = []
    lines.append(r"\begin{table}[t]")
    lines.append(r"  \centering")
    lines.append(r"  \small")
    lines.append(r"  \caption{Generation efficiency by experimental factor (mean $\pm$ SD), ordered by tokens. \textbf{Bold} = best per factor. N varies by design: CoT excludes DeepSeek-R1; SARIMAX uses only ``none'' XAI.}")
    lines.append(r"  \label{tab:efficiency}")
    lines.append("")
    lines.append(r"  \begin{tabular}{@{}llcrr@{}}")
    lines.append(r"    \toprule")
    
    # Header with grand mean
    gn, gt, gts, gti, gtis = grand_stats
    lines.append(f"    \\textbf{{Factor}} & \\textbf{{Level}} & \\textbf{{N}} & \\textbf{{Tokens}} ({gt:.0f}$\\pm${gts:.0f}) & \\textbf{{Time (s)}} ({gti:.1f}$\\pm${gtis:.1f}) \\\\")
    lines.append(r"    \midrule")
    
    current_factor = None
    
    for _, row in efficiency_df.iterrows():
        factor = row['Factor']
        level = str(row['Level'])
        n = int(row['N'])
        tokens_mean = int(row['Tokens_mean'])
        tokens_std = int(row['Tokens_std'])
        time_mean = row['Time_mean']
        time_std = row['Time_std']
        
        # Clean level name
        level_display = level_names.get(level, level)
        
        # Find best in factor
        factor_data = efficiency_df[efficiency_df['Factor'] == factor]
        best_tokens = factor_data['Tokens_mean'].min()
        best_time = factor_data['Time_mean'].min()
        
        # Format with bold for best
        if tokens_mean == int(best_tokens):
            tokens_str = f"\\textbf{{{tokens_mean}}}\\tiny$\\pm${tokens_std}"
        else:
            tokens_str = f"{tokens_mean}\\tiny$\\pm${tokens_std}"
        
        if time_mean == best_time:
            time_str = f"\\textbf{{{time_mean:.1f}}}\\tiny$\\pm${time_std:.1f}"
        else:
            time_str = f"{time_mean:.1f}\\tiny$\\pm${time_std:.1f}"
        
        # Add factor label with multirow
        if factor != current_factor:
            if current_factor is not None:
                lines.append(r"    \midrule")
            factor_count = len(factor_data)
            factor_label = factor_labels.get(factor, factor)
            lines.append(f"    \\multirow{{{factor_count}}}{{*}}{{\\rotatebox{{90}}{{\\textbf{{{factor_label}}}}}}}")
            current_factor = factor
        
        lines.append(f"      & {level_display:17} & {n:4} & {tokens_str:30} & {time_str} \\\\")
    
    lines.append(r"    \bottomrule")
    lines.append(r"  \end{tabular}")
    lines.append(r"\end{table}")
    
    return "\n".join(lines)

# Generate complete LaTeX table
grand_stats = (grand_n, grand_tokens_mean, grand_tokens_std, grand_time_mean, grand_time_std)
latex_table = generate_full_latex_efficiency_table(efficiency_table, grand_stats)

print("Complete LaTeX Efficiency Table:")
print("=" * 80)
print(latex_table)

Complete LaTeX Efficiency Table:
\begin{table}[t]
  \centering
  \small
  \caption{Generation efficiency by experimental factor (mean $\pm$ SD), ordered by tokens. \textbf{Bold} = best per factor. N varies by design: CoT excludes DeepSeek-R1; SARIMAX uses only ``none'' XAI.}
  \label{tab:efficiency}

  \begin{tabular}{@{}llcrr@{}}
    \toprule
    \textbf{Factor} & \textbf{Level} & \textbf{N} & \textbf{Tokens} (653$\pm$1188) & \textbf{Time (s)} (35.2$\pm$72.0) \\
    \midrule
    \multirow{8}{*}{\rotatebox{90}{\textbf{Strategy}}}
      & CoT Few           &   60 & \textbf{215}\tiny$\pm$39       & 35.2\tiny$\pm$34.0 \\
      & CoT Zero          &   60 & 279\tiny$\pm$45                & 29.1\tiny$\pm$27.6 \\
      & Zero-shot         &   90 & 498\tiny$\pm$618               & \textbf{22.9}\tiny$\pm$23.7 \\
      & Few-shot          &   90 & 520\tiny$\pm$743               & 29.5\tiny$\pm$31.8 \\
      & Meta-prompt       &   90 & 525\tiny$\pm$435               & 29.4\tiny$\pm$32.3 \\
     